# Polynomial regression

A quadratic polynomial is still a linear regression. Linearity refers to the coefficients, not to the shape of the curve in $x$. The design matrix carries columns $1$, $x$, and $x^2$, and the normal equations are unchanged.


## An exact quadratic

Let $x = 0, 1, 2, 3$ and $y = (x - 1)^2$, so

$$
y = 1,\ 0,\ 1,\ 4
$$

and the algebraic expansion is $y = 1 - 2x + x^2$. Substituting the coefficient vector $(1, -2, 1)^{\mathsf T}$ into every row gives

$$
\begin{aligned}
1 - 2\cdot 0 + 0^2 &= 1, \\
1 - 2\cdot 1 + 1^2 &= 0, \\
1 - 2\cdot 2 + 2^2 &= 1, \\
1 - 2\cdot 3 + 3^2 &= 4.
\end{aligned}
$$

The residual sum of squares is zero. A residual sum of squares cannot be negative, so this coefficient vector is a least-squares solution. It is the only one if the three columns $1$, $x$, and $x^2$ are linearly independent, which the nonzero determinant of $X^{\mathsf T}X$ confirms in the code. Therefore ordinary least squares returns exactly $(1, -2, 1)$.


## The straight line on the same points

A model that may use only $1$ and $x$ cannot represent $x^2$. Its sums are

$$
\begin{aligned}
\bar x &= \dfrac{3}{2}, & \bar y &= \dfrac{3}{2}, &
S_{xx} &= 5, & S_{xy} &= 5,
\end{aligned}
$$

so $b_1 = 1$ and $b_0 = 0$. The predictions $0, 1, 2, 3$ leave residuals $1, -1, -1, 1$ and

$$
\mathrm{RSS} = 4, \qquad R^2 = 1 - \dfrac{4}{9} = \dfrac{5}{9}.
$$

The quadratic fit has $\mathrm{RSS} = 0$ on the same four points. The comparison is fair only because both fits were scored by the residual sum of squares. The quadratic spent one more coefficient to buy that zero.


In [1]:
# The quadratic is an exact fit. The straight line leaves RSS 4.
from fractions import Fraction
import numpy as np
x_list = [0, 1, 2, 3]
y_list = [(xi - 1) ** 2 for xi in x_list]
x = np.array(x_list, dtype=float)
y = np.array(y_list, dtype=float)
quadratic = np.column_stack([np.ones(4), x, x ** 2])
beta_q = np.linalg.solve(quadratic.T @ quadratic, quadratic.T @ y)
n = 4
sx, sy = sum(x_list), sum(y_list)
sxy = sum(a * b for a, b in zip(x_list, y_list))
sx2 = sum(a * a for a in x_list)
slope = Fraction(n * sxy - sx * sy, n * sx2 - sx ** 2)
intercept = Fraction(sy - slope * sx, n)
line_rss = sum((yi - (intercept + slope * xi)) ** 2 for xi, yi in zip(x_list, y_list))
tss = sum((yi - Fraction(sy, n)) ** 2 for yi in y_list)
print("quadratic beta", beta_q)
print("line", intercept, slope, "RSS", line_rss, "R2", 1 - line_rss / tss)
assert np.allclose(beta_q, [1, -2, 1])
assert np.sum((y - quadratic @ beta_q) ** 2) < 1e-12
assert intercept == 0 and slope == 1 and line_rss == 4
assert 1 - line_rss / tss == Fraction(5, 9)


quadratic beta [ 1. -2.  1.]
line 0 1 RSS 4 R2 5/9


## One change: move the last response by one

Replace $y$ by $1, 0, 1, 5$. The vector is no longer an exact quadratic. Solving the three normal equations produces

$$
b_0 = \dfrac{21}{20}, \qquad b_1 = -\dfrac{49}{20}, \qquad b_2 = \dfrac{5}{4}, \qquad \mathrm{RSS} = \dfrac{1}{20}.
$$

A cubic column $x^3$ can be added. Four coefficients and four distinct $x$ values make a square Vandermonde matrix of full rank, so there is a unique interpolant and its residual sum of squares is zero. That interpolant is

$$
1 - \dfrac{5}{3}x + \dfrac{1}{2}x^2 + \dfrac{1}{6}x^3.
$$

A residual sum of squares of zero now means that the model had enough flexibility to pass through every point. It does not mean the cubic was the mechanism that generated the table.


In [2]:
# A cubic can interpolate four points that are not a quadratic.
from fractions import Fraction
import numpy as np
x = np.array([0.0, 1, 2, 3])
y = np.array([1.0, 0, 1, 5])
quadratic = np.column_stack([np.ones(4), x, x ** 2])
cubic = np.column_stack([quadratic, x ** 3])
beta_q = np.linalg.solve(quadratic.T @ quadratic, quadratic.T @ y)
beta_c = np.linalg.solve(cubic.T @ cubic, cubic.T @ y)
rss_q = np.sum((y - quadratic @ beta_q) ** 2)
rss_c = np.sum((y - cubic @ beta_c) ** 2)
print("quadratic", beta_q, "RSS", rss_q)
print("cubic", beta_c, "RSS", rss_c)
assert np.allclose(beta_q, [21 / 20, -49 / 20, 5 / 4])
assert abs(rss_q - 0.05) < 1e-12
assert np.allclose(beta_c, [1, -5 / 3, 1 / 2, 1 / 6])
assert rss_c < 1e-12


quadratic [ 1.05 -2.45  1.25] RSS 0.049999999999999975
cubic [ 1.         -1.66666667  0.5         0.16666667] RSS 6.025541461207682e-28


## Pitfall

Calling a polynomial "nonlinear regression" because its graph bends confuses the shape in $x$ with the algebra in $\beta$. The loss $S(\beta) = \|y - X\beta\|^2$ is still quadratic in $\beta$, and the normal equations are still linear. A model becomes nonlinear regression when a parameter sits inside a power, an exponential, or another function that cannot be written as a column of $X$ known before $\beta$ is estimated. $y = b_0 + b_1 x + b_2 x^2$ is linear in $(b_0, b_1, b_2)$. $y = b_0 + e^{b_1 x}$ is not.


In [3]:
# x squared is a known column. An exponential coefficient would not be.
import numpy as np
x = np.array([0.0, 1, 2, 3])
column = x ** 2
print("known before any coefficient is estimated", column)
assert np.array_equal(column, np.array([0.0, 1, 4, 9]))


known before any coefficient is estimated [0. 1. 4. 9.]


## Summary

- Columns $1$, $x$, and $x^2$ fit a quadratic by ordinary least squares.
- On $y = (x-1)^2$ the fit is exactly $1 - 2x + x^2$. The straight line leaves $\mathrm{RSS} = 4$.
- Four points and a cubic give $\mathrm{RSS} = 0$ whether or not the cubic is the true shape. Perfect fit can mean "no residual degrees of freedom left."
